Idea: finitune mobilenet_v3_large to produce probabilities of sides (5 categories), probability of damage (2 categories) and quality of photo (2 categories), total 9 categories

this is all done via summing crossentropylosses on those three categories

then we train a logreg head on matrix of 4x9 and adding polynomial features, matrix we get from 4 photos, each 9 logits

In [1]:
import copy
import csv
from enum import Enum
import io
import json
import os
import typing as t

#import cv2
from IPython.display import clear_output
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    roc_curve,
)
import torch
import torch.nn as nn
from torch.nn import functional as F
import torch.utils.data as td
from torch.utils.data import DataLoader, Dataset
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights, mobilenet_v3_large

%load_ext autoreload
%autoreload 2

In [2]:
from utils_fixed_1 import (
    show_photos, 
    #create_dataloader,
    train_epoch,
    test_epoch,
    plot_history,
    print_model_params_required_grad,
    PUBLIC_DATA_FOLDER_PATH,
    PUBLIC_DATA_DESCRIPTION_PATH,
)
from utils import create_dataloader

In [3]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda', index=0)

# Description

Yandex GO is one of the top three ride-hailing services in the world. Our app facilitates over 4 billion trips per year across 32 countries. We are committed to the quality of our services, ensuring thorough checks of both drivers and their vehicles before they go online, based on dozens of criteria. Part of the vehicle inspection process is carried out remotely using photos of the vehicle, which allows us to either block or grant the driver access to orders. This tool ensures that cars do not go online if they are damaged or dirty.

Computer vision algorithms play a significant role in this remote quality control process. Machine learning models act as a filter that processes vehicle inspection requests, automatically approving a portion of requests that, according to the models, contain no violations, and sending suspicious cases for additional manual review.

### How does the photo inspection process work?
As part of vehicle photo inspections, drivers periodically receive a task to take photos of their car, so it can be checked for damage, compliance with service standards, branding presence, etc. Before these checks, we also need to ensure that drivers took the photos honestly and sent what we expected. The driver is required to take 4 photos (front, rear, left side, right side). The photos are taken through the Yandex PRO app, which has an interface that guides them to capture the 4 photos in the correct order and from the required angles.

In the standard process, the photos are first reviewed by ML pipeline. If ML pipeline doesn't find anything suspicious in the photos, the inspection is automatically approved. If the pipeline flags at least one photo, the inspection is sent to an assessor for a final decision. Thus, the object for decision-making is the inspection itself, i.e., all 4 photos together.

In this task, the license plate numbers have been blacked out.

In [ ]:
pass_id = '000f43a6549ad26d'
photos = []
for side in ['front', 'back', 'left', 'right']:
    with open(f'{PUBLIC_DATA_FOLDER_PATH}/{pass_id}_{side}', 'rb') as file:
        photos.append(file.read())
show_photos(photos)

### Data description: 
- **filename** —  name of the photo file, consisting of `pass_id` and `plan_side`.
- **pass_id** — ID of the inspection. Each inspection contains 4 photos.
- **plan_side** — the side of the vehicle that should be in the photo. Possible values: front, back, left, right.
- **fact_side** — the side of the vehicle as determined by assessors. Possible values: front, back, left, right, unknown.
- **fraud_verdict** — the assessor's verdict on what is depicted in the photo. Possible values:
   - ALL_GOOD —  the photo clearly shows one side of the vehicle, which is fully visible and in focus.
   - LACK_OF_PHOTOS — the photo does not contain a vehicle at all.
   - BLURRY_PHOTO — the photo is blurry.
   - SCREEN_PHOTO — not a real vehicle photo, but a photo of a screen.
   - DARK_PHOTO — the photo is too dark.
   - INCOMPLETE_CAPTURE — the vehicle is not fully visible in the photo.
- **fraud_probability** — the proportion of assessors who assigned the given fraud_verdict. If no verdict achieved a majority, a random one is chosen.
- **damage_verdict** — the assessor's verdict on the vehicle's condition. Possible values:
   - NO_DEFECT —  no visible damage.
   - DEFECT — the is some damage.
   - BAD_PHOTO — can't say anything about the damage, because of photo's quality.
- **damage_probability** — the proportion of assessors who assigned the given damage_verdict. If no verdict achieved a majority, a random one is chosen.

In [8]:
description = pd.read_csv(PUBLIC_DATA_DESCRIPTION_PATH, index_col='filename').sort_index()
description.head()

,pass_id,plan_side,fact_side,fraud_verdict,fraud_probability,damage_verdict,damage_probability
filename,,,,,,,
00015b960a1c013e_back,00015b960a1c013e,back,back,DARK_PHOTO,1.000000,BAD_PHOTO,0.8
00015b960a1c013e_front,00015b960a1c013e,front,front,DARK_PHOTO,0.666667,BAD_PHOTO,1.0
00015b960a1c013e_left,00015b960a1c013e,left,unknown,DARK_PHOTO,0.666667,BAD_PHOTO,0.6
00015b960a1c013e_right,00015b960a1c013e,right,unknown,DARK_PHOTO,0.666667,BAD_PHOTO,1.0
0001f673ef360c58_back,0001f673ef360c58,back,back,ALL_GOOD,0.666667,NO_DEFECT,1.0


In addition to fraud that can be identified by looking at an individual photo, there may be cases where each photo individually has a fraud_verdict of 'ALL_GOOD', but the driver took two photos of the same side of the vehicle and failed to capture another side:

In [12]:
description[description.pass_id == '001c07aa1e3edf7e']

,pass_id,plan_side,fact_side,fraud_verdict,fraud_probability,damage_verdict,damage_probability
filename,,,,,,,
001c07aa1e3edf7e_back,001c07aa1e3edf7e,back,back,ALL_GOOD,1.0,NO_DEFECT,1.0
001c07aa1e3edf7e_front,001c07aa1e3edf7e,front,front,ALL_GOOD,1.0,NO_DEFECT,1.0
001c07aa1e3edf7e_left,001c07aa1e3edf7e,left,front,ALL_GOOD,1.0,NO_DEFECT,1.0
001c07aa1e3edf7e_right,001c07aa1e3edf7e,right,back,ALL_GOOD,1.0,NO_DEFECT,0.8


# Objective
To assess the quality of vehicles and photos using machine learning algorithms:  
1. For detecting fraud (incorrect photos, unclear images, or incorrect photo sets).  
2. For detecting vehicle damage.

## Performance Metric and Deliverables
There are 2 targets and 4 sides of a vehicle in each exam. But after all, we need to predict whether the inspection should be sent to a human for review to provide feedback to the driver, or if there are no defects and the inspection can be automatically approved. This means that the metric is calculated not for individual photos for each target, but for the inspection as a whole.

*Evaluation Metric:* ROC AUC (object — inspection)

*Required Deliverables*:
- Model Weights: The trained model's weights for reproducibility and further analysis.
- Executable Script: A script containing all necessary code to run the model, including data reading, preprocessing steps, model architecture, inference code.
   

# Example

Let's try to train a fraud detection model with a simplified target that does not account for cases where two photos in an inspection may capture the same side of the vehicle. For this, we will use a pretrained ResNet18 model and replace its classifier.

In [13]:
class CarSide(Enum):
    FRONT = 0
    BACK = 1
    LEFT = 2
    RIGHT = 3
    UNKNOWN = 5
    
class FraudResolution(Enum):
    ALL_GOOD = 0
    LACK_OF_PHOTOS = 1
    BLURRY_PHOTO = 2
    SCREEN_PHOTO = 3
    DARK_PHOTO = 4
    INCOMPLETE_CAPTURE = 5
    RUDE_CONTENT = 6
    
class DamageResolution(Enum):
    NO_DEFECT = 0
    DEFECT = 1
    BAD_PHOTO = 2

In [14]:
IMAGENET_RGB_MEAN = [0.485, 0.456, 0.406]
IMAGENET_RGB_STD = [0.229, 0.224, 0.225]
RESIZE_SIZE = (256, 256)


def pil_open(image_data: bytes) -> Image:
    return Image.open(io.BytesIO(image_data))


def preprocess(image_data: t.Optional[bytes]) -> torch.Tensor:
    return transforms.Compose([
        transforms.Lambda(pil_open),
        transforms.ToTensor(),
        transforms.Resize(RESIZE_SIZE),
        transforms.Normalize(IMAGENET_RGB_MEAN, IMAGENET_RGB_STD),
    ])(image_data)

In [15]:
def get_damage_target(damage_resolution, *args):
    return int(damage_resolution != DamageResolution.NO_DEFECT.name)

In [16]:
def get_target_vector(fact_side, fraud_probability, fraud_verdict, damage_probability, damage_verdict):

    class_id = -1
    if fact_side == 'front':
        class_id = 0
    elif fact_side == 'back':
        class_id = 1
    elif fact_side == 'left':
        class_id = 2
    elif fact_side == 'right':
        class_id = 3
    elif fact_side == 'unknown':
        class_id = 4
    #return class_id
    one_hot = np.zeros(5)
    one_hot[class_id] = 1
    if class_id == -1:
        return np.zeros(5)
    p_fraud = fraud_probability*int(fraud_verdict == 'ALL_GOOD')+\
                (1-fraud_probability)*(1-int(fraud_verdict == 'ALL_GOOD'))
    
    p_damage = damage_probability*int(damage_verdict == 'NO_DEFECT')+\
                (1-damage_probability)*(1-int(damage_verdict == 'NO_DEFECT'))
    ground_truth = int((damage_verdict == 'NO_DEFECT') and (fraud_verdict == 'ALL_GOOD'))
    return np.hstack((one_hot, np.array([p_fraud, p_damage, ground_truth])))

In [17]:
def get_side_idx(fact_side):
    class_id = -1
    if fact_side == 'front':
        class_id = 0
    elif fact_side == 'back':
        class_id = 1
    elif fact_side == 'left':
        class_id = 2
    elif fact_side == 'right':
        class_id = 3
    elif fact_side == 'unknown':
        class_id = 4
    return class_id

In [18]:
target_vector = []
for pass_id, filename, fact_side, fraud_probability, fraud_verdict, damage_probability, damage_verdict in zip( description.pass_id, description.index, description.fact_side, description.fraud_probability, description.fraud_verdict, description.damage_probability, description.damage_verdict):
    
    target_vector.append({'pass_id': pass_id, 'filename': filename, 'target': get_target_vector(fact_side, fraud_probability, fraud_verdict, damage_probability, damage_verdict)})
target_vector = pd.DataFrame(target_vector)

In [19]:
target_vector.head()

,pass_id,filename,target
0,00015b960a1c013e,00015b960a1c013e_back,"[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.1999999999999..."
1,00015b960a1c013e,00015b960a1c013e_front,"[1.0, 0.0, 0.0, 0.0, 0.0, 0.33333333333333337,..."
2,00015b960a1c013e,00015b960a1c013e_left,"[0.0, 0.0, 0.0, 0.0, 1.0, 0.33333333333333337,..."
3,00015b960a1c013e,00015b960a1c013e_right,"[0.0, 0.0, 0.0, 0.0, 1.0, 0.33333333333333337,..."
4,0001f673ef360c58,0001f673ef360c58_back,"[0.0, 1.0, 0.0, 0.0, 0.0, 0.6666666666666666, ..."


In [20]:
target_vector.set_index('filename', inplace=True)

In [21]:
target_vector.head()

,pass_id,target
filename,,
00015b960a1c013e_back,00015b960a1c013e,"[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.1999999999999..."
00015b960a1c013e_front,00015b960a1c013e,"[1.0, 0.0, 0.0, 0.0, 0.0, 0.33333333333333337,..."
00015b960a1c013e_left,00015b960a1c013e,"[0.0, 0.0, 0.0, 0.0, 1.0, 0.33333333333333337,..."
00015b960a1c013e_right,00015b960a1c013e,"[0.0, 0.0, 0.0, 0.0, 1.0, 0.33333333333333337,..."
0001f673ef360c58_back,0001f673ef360c58,"[0.0, 1.0, 0.0, 0.0, 0.0, 0.6666666666666666, ..."


In [22]:
from torchvision.transforms import RandomApply, RandomRotation, ColorJitter

In [23]:
# Define advanced data augmentation
def advanced_preprocess(image_data: t.Optional[bytes]) -> torch.Tensor:
    return transforms.Compose([
        transforms.Lambda(pil_open),
        transforms.Resize(RESIZE_SIZE),
        transforms.RandomHorizontalFlip(),
        RandomApply([RandomRotation(degrees=15)], p=0.5),
        RandomApply([ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1)], p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_RGB_MEAN, IMAGENET_RGB_STD),
    ])(image_data)

In [24]:
target_vector.head()

,pass_id,target
filename,,
00015b960a1c013e_back,00015b960a1c013e,"[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.1999999999999..."
00015b960a1c013e_front,00015b960a1c013e,"[1.0, 0.0, 0.0, 0.0, 0.0, 0.33333333333333337,..."
00015b960a1c013e_left,00015b960a1c013e,"[0.0, 0.0, 0.0, 0.0, 1.0, 0.33333333333333337,..."
00015b960a1c013e_right,00015b960a1c013e,"[0.0, 0.0, 0.0, 0.0, 1.0, 0.33333333333333337,..."
0001f673ef360c58_back,0001f673ef360c58,"[0.0, 1.0, 0.0, 0.0, 0.0, 0.6666666666666666, ..."


we gropup cliues of same pass_id and feed them to dataloaders for train and test

In [25]:
BATCH_SIZE = 64
TRAIN_FRACTION = 0.7
subset = 1
total_size = int(target_vector.shape[0] / subset)
train_size = int(total_size * TRAIN_FRACTION)
pass_ids = pd.unique(target_vector.pass_id)

random_separation = np.random.rand(len(pass_ids))
random_pass_id = []



train_pd = []
test_pd = []
cnt = 0
for pass_id, group in target_vector.groupby("pass_id"):
    cnt+=1
    if cnt<0.7*total_size/4: 
        train_pd.append(group)
    else:
        test_pd.append(group)

train_pd = pd.concat(train_pd)
test_pd = pd.concat(test_pd)

In [27]:

train_loader = create_dataloader(
    img_dir_path=PUBLIC_DATA_FOLDER_PATH,
    target_map=train_pd.target.to_dict(),
    description=description,
    batch_size=BATCH_SIZE,
    preprocessor=preprocess,
    num_load_workers=10,
)

test_loader = create_dataloader(
    img_dir_path=PUBLIC_DATA_FOLDER_PATH,
    target_map=test_pd.target.to_dict(),
    description=description,
    batch_size=BATCH_SIZE,
    preprocessor=preprocess,
    num_load_workers=10,
)

In [28]:
def train_model(
    model, 
    device, 
    train_loader, 
    test_loader, 
    epochs, 
    criterion, 
    optimizer, 
    scheduler=None, 
    save_best_model=True
):
    best_test_loss = None
    best_state_dict = None
    
    train_loss_history = []
    train_sides_acc_history = []
    train_fraud_acc_history = []
    train_damage_acc_history = []
    
    test_loss_history = []
    test_sides_acc_history = []
    test_fraud_acc_history = []
    test_damage_acc_history = []
    
    model = model.to(device)
    
    for epoch in range(epochs):
        print(f'Epoch {epoch + 1}')
        train_loss, train_sides_acc, train_fraud_acc, train_damage_acc = train_epoch(
            model, 
            device,
            train_loader, 
            criterion, 
            optimizer
        )
        train_loss_history.append(train_loss)
        train_sides_acc_history.append(train_sides_acc)
        train_fraud_acc_history.append(train_fraud_acc)
        train_damage_acc_history.append(train_damage_acc)

        if scheduler is not None:
            scheduler.step()
        
        test_loss, test_sides_acc, test_fraud_acc, test_damage_acc = test_epoch(model, device, test_loader, criterion)
        test_loss_history.append(test_loss)
        test_sides_acc_history.append(test_sides_acc)
        test_fraud_acc_history.append(test_fraud_acc)
        test_damage_acc_history.append(test_damage_acc)
        
        if best_test_loss is None or test_loss < best_test_loss:
            best_test_loss = test_loss
            best_state_dict = copy.deepcopy(model.state_dict())
        
        clear_output()
        plot_history(
            train_loss_history,
            train_sides_acc_history ,
            train_fraud_acc_history ,
            train_damage_acc_history ,
            test_loss_history ,
            test_sides_acc_history,
            test_fraud_acc_history,
            test_damage_acc_history
        )
    
    if save_best_model:
        model.load_state_dict(best_state_dict)
    
    return {
        'train_loss': train_loss_history, 
        'test_loss': test_loss_history,
        'train_acc': train_acc_history,
        'test_acc': test_acc_history
    }

### MobileNetV3

In [29]:
model = mobilenet_v3_large(pretrained=False)
model.load_state_dict(torch.load("../models/mobilenet_v3_large-5c1a4163.pth"))

/opt/software/python/envs/google_colab_gpu_2024/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/software/python/envs/google_colab_gpu_2024/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
<ipython-input-29-7341114d9a07>:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default val

<All keys matched successfully>

unfreeze backbone

In [31]:
# load pretrained model
#model = resnet18(pretrained=False)


# replace classifier
model.classifier = torch.nn.Sequential(
    torch.nn.Linear(960, 256),
    torch.nn.BatchNorm1d(256),
    torch.nn.ReLU(),
    torch.nn.Dropout(0.3),
    torch.nn.Linear(256, 64),
    torch.nn.BatchNorm1d(64),
    torch.nn.ReLU(),
    torch.nn.Linear(64, 9), #9 categories

)

In [32]:
!nvidia-smi

Mon Dec  2 18:20:22 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.86.10              Driver Version: 535.86.10    CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla V100-SXM2-32GB           On  | 00000000:1D:00.0 Off |                    0 |
| N/A   29C    P0              55W / 300W |   4225MiB / 32768MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

sum of 3 crossentropy losses with weights, 

In [33]:
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5, weight_decay=1e-3)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
class_weights = torch.tensor([0.5, 0.5], device=device)

def loss_7d(weights_fraud, weights_damage, weights_loss):
    
    def bce_loss(pred, target, weights):
        pred = torch.clamp(pred, min=1e-7, max=1-1e-7)
        bce = -weights[1] * target * torch.log(pred) - (1 - target) * weights[0] * torch.log(1 - pred)
        return torch.sum(bce)
    sigmoid = torch.nn.Sigmoid()
    bce = torch.nn.BCELoss()
    return lambda prd, trg: torch.nn.CrossEntropyLoss()(prd[:, :5], trg[:, :5])+\
                            weights_loss[0]*torch.nn.CrossEntropyLoss()(prd[:, 5:7], torch.vstack((trg[:, 5], 1-trg[:, 5])).T)+\
                            weights_loss[1]*torch.nn.CrossEntropyLoss()(prd[:, 7:9], torch.vstack((trg[:, 6], 1-trg[:, 6])).T)
                   
criterion = loss_7d(weights_fraud = [0.5, 0.5], weights_damage = [0.5, 0.5], weights_loss = [1.34, 1.34])

In [34]:
a = [1, 2, 3, 4, 5, 6, 7, 8, 9]

In [35]:
a[5:7], a[7:9]

([6, 7], [8, 9])

In [ ]:
enhanced_model_log = train_model(
    model=model, 
    device=device,
    train_loader=train_loader, 
    test_loader=test_loader, 
    epochs=8,  # Train for more epochs
    criterion=criterion, 
    optimizer=optimizer,
    scheduler=lr_scheduler
)

In [36]:
torch.save(model, 'roma_7d_v6.pt')

In [37]:
model = torch.load('roma_7d_v5.pt')

<ipython-input-37-95f28bc71ce6>:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load('roma_7d_v5.pt')


mark data for training logreg

In [ ]:
from utils_fixed_1 import mark_data_from_one_model


model = torch.load('roma_7d_v5.pt')
feature_extraction_train = mark_data_from_one_model(model, device, train_loader)
feature_extraction_test = mark_data_from_one_model(model, device, test_loader)

messy pandas manypulations

In [ ]:
feature_extraction_train

In [ ]:
feature_extraction_train

In [75]:
new_train_data = []
dimens = 9
for pass_id, group in feature_extraction_train.groupby("pass_id"):
    prediction_list = list(group['prediction'])+[np.zeros(dimens) for i in range(4-len(group))]

    side_truth = np.prod([ get_side_idx(i)==j for i, j in zip(list(group['plan_side']), list(group['fact_side']))])
    new_train_data.append([pass_id, np.asarray(prediction_list).reshape(dimens*4), (len(group)==4)*group['ground_truth'].max()*side_truth])
new_train_data_df = pd.DataFrame(new_train_data, columns=['pass_id', 'prediction', 'ground_truth'])


In [76]:
new_test_data = []
for pass_id, group in feature_extraction_test.groupby("pass_id"):
    prediction_list = list(group['prediction'])+[np.zeros(dimens) for i in range(4-len(group))]
    side_truth = np.prod([ get_side_idx(i)==j for i, j in zip(list(group['plan_side']), list(group['fact_side']))])
    new_test_data.append([pass_id, np.asarray(prediction_list).reshape(dimens*4), (len(group)==4)*group['ground_truth'].max()*side_truth])
new_test_data_df = pd.DataFrame(new_test_data, columns=['pass_id', 'prediction', 'ground_truth'])


In [77]:
feature_extraction_train.set_index('pass_id', inplace = True)

In [78]:
feature_extraction_train.iloc[:20000]


,prediction,plan_side,fact_side,ground_truth
pass_id,,,,
00015b960a1c013e,"[-1.620479, 2.4038343, -2.15616, -1.8340611, -...",back,1,0.0
00015b960a1c013e,"[0.6280802, -1.089518, -0.9152178, -0.8174122,...",front,0,0.0
00015b960a1c013e,"[-1.672443, -1.7732869, 0.16992086, 0.90404934...",left,0,0.0
00015b960a1c013e,"[-1.288993, -1.6582884, -0.67098397, -0.299933...",right,0,0.0
0001f673ef360c58,"[-1.1223993, 3.1176777, -1.7536886, -1.6655054...",back,1,1.0
...,...,...,...,...
19f57974451857c6,"[-2.0499644, -1.8852016, -0.47179908, 2.429185...",right,3,0.0
19f57cf2562f8468,"[-0.35535014, 3.0013862, -1.9022524, -2.135676...",back,1,0.0
19f57cf2562f8468,"[3.5998528, -1.715654, -2.0841331, -1.4353931,...",front,0,1.0


get real target

In [79]:
def make_complex_target(row):
    real = int(
        row.plan_side != row.fact_side or 
        row.fraud_verdict != 'ALL_GOOD' or
        row.damage_verdict != 'NO_DEFECT'
    )
    return pd.Series(
        data=[row.pass_id, real],
        index=['pass_id', 'real'],
    )

In [80]:
real_truth = description.apply(make_complex_target, axis = 1)

In [81]:
new_train_data_df.set_index('pass_id', inplace=True)

In [82]:
real_truth2 = real_truth.set_index('pass_id')

In [83]:
real_truth3 = real_truth2.groupby('pass_id').max()

In [84]:
new_train_data_df['prediction'].iloc[0].shape

(36,)

gridsearch to find hyperparametres of logreg

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

clf = LogisticRegression(solver='newton-cg')
dimens = 9

X_train = np.zeros((len(new_train_data_df), dimens*8+(dimens*4)**2))
y_train =  np.zeros((len(new_train_data_df)))

for idx, i, j in zip(range(len(new_train_data_df)), new_train_data_df.prediction, real_truth3.iloc[:len(new_train_data_df)].real):
    X_train[idx, :] = np.hstack((i,(i[None,:]*i[:, None]).flatten(), (i**3).flatten()))
    y_train[idx] = j
    
grid = GridSearchCV(LogisticRegression(random_state=42),
    param_grid={"C": [50.0, 20.0, 10.0, 1.1, 1.0, 0.8, 0.5, 0.001, 0.0002], "penalty": ["l2"]})
grid.fit(X_train, y_train)

train logreg with optimal hyperparametres

In [ ]:

clf = LogisticRegression(C=0.001, random_state=42)
clf.fit(X_train, y_train)

check our roc_auc

In [87]:
from sklearn.metrics import roc_auc_score
X_test = np.zeros((len(new_test_data_df), dimens*8+(dimens*4)**2))
y_test =  np.zeros((len(new_test_data_df)))
for idx, i, j in zip(range(len(new_test_data_df)), new_test_data_df.prediction, real_truth3.iloc[len(new_train_data_df):].real):
    X_test[idx, :] =  np.hstack((i,(i[None,:]*i[:, None]).flatten(), (i**3).flatten()))
    y_test[idx] = j
y_pred = clf.predict_proba(X_test)[:, 1]
roc_auc_score(y_test, y_pred)


0.9816020844829116

In [ ]:
from sklearn.metrics import roc_auc_score
X_test = np.zeros((len(new_test_data_df), 28))
y_test =  np.zeros((len(new_test_data_df)))
for idx, i, j in zip(range(len(new_test_data_df)), new_test_data_df.prediction, real_truth3.iloc[len(new_train_data_df):].real):
    X_test[idx, :] = i
    y_test[idx] = j
y_pred = []
for idx in range(len(X_test)):
    x = X_test[idx, :].reshape((4, 7))
    y_pred.append(np.prod(np.diag(x))+np.prod(x[:, -1]))
roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1])

roc_auc_score(y_test, y_pred)


In [88]:
import pickle

with open("clf_98.pickle", "wb") as f:
    pickle.dump(clf, f)

### Metric calculation

**NB**: There are only **filename**, **pass_id**, **plan_side** in private data description

In [25]:
from sklearn.metrics import roc_auc_score

from utils import get_predictions

In [26]:
model = torch.load('mobilenetv3_large_damage.pt')
model.to(device)

<ipython-input-26-785549286dd2>:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load('mobilenetv3_large_damage.pt')


MobileNetV3(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        )
      )
    )
    (2): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 64, kernel_size=(1, 1), stride=(1, 1), bi

In [27]:
test_predictions = get_predictions(model, device, test_loader)

  0%|          | 0/853 [00:14<?, ?it/s]

In [28]:
test_predictions.head()

,pass_id,prediction,plan_side
0,a1240a46c165f6ca,0.998828,back
1,a1240a46c165f6ca,0.998929,front
2,a1240a46c165f6ca,0.998513,left
3,a1240a46c165f6ca,0.992907,right
4,a1252381f49a5101,0.019021,back


In [31]:
def make_complex_target(row):
    real = int(
        row.plan_side != row.fact_side or 
        row.fraud_verdict != 'ALL_GOOD' or
        row.damage_verdict != 'NO_DEFECT'
    )
    return pd.Series(
        data=[row.pass_id, real, row.prediction],
        index=['pass_id', 'real', 'pred'],
    )

# All predictions for each vehicle are aggregated into a single value, 
# and the metric is calculated based on the inspections.
test_verdicts = test_predictions.merge(
    description, 
    on=['pass_id', 'plan_side']
).apply(make_complex_target, axis=1).groupby('pass_id').max()

test_verdicts.head()

,real,pred
pass_id,,
a1240a46c165f6ca,1,0.998929
a1252381f49a5101,1,0.143458
a1256f6b65a2193b,1,0.943684
a12659d9dceef2aa,1,0.757587
a1289c09d5a573cc,1,0.951786


In [32]:
score = roc_auc_score(test_verdicts.real, test_verdicts.pred)
print(f'simple fraud target roc_auc_score: {score}')

simple fraud target roc_auc_score: 0.8879176927747199


### Make a submission file

In [96]:
solution_script = '''
import typing as t
import io

import pandas as pd
from PIL import Image
import numpy as np
import torch
import torchvision.transforms as transforms
from torchvision.models import mobilenet_v3_large
from tqdm.auto import tqdm
from sklearn.linear_model import LogisticRegression

import pickle

from utils_fixed import (
    get_predictions, 
    create_dataloader,
    PRIVATE_DATA_FOLDER_PATH, 
    PRIVATE_DATA_DESCRIPTION_PATH,
)

BATCH_SIZE = 64
IMAGENET_RGB_MEAN = [0.485, 0.456, 0.406]
IMAGENET_RGB_STD = [0.229, 0.224, 0.225]
RESIZE_SIZE = (256, 256)


def pil_open(image_data: bytes) -> Image:
    return Image.open(io.BytesIO(image_data))


def preprocess(image_data: t.Optional[bytes]) -> torch.Tensor:
    return transforms.Compose([
        transforms.Lambda(pil_open),
        transforms.ToTensor(),
        transforms.Resize(RESIZE_SIZE),
        transforms.Normalize(IMAGENET_RGB_MEAN, IMAGENET_RGB_STD),
    ])(image_data)

device = torch.device('cpu')
model = mobilenet_v3_large(pretrained=False)
model = torch.load('roma_7d_v5.pt', map_location=device)

description = pd.read_csv(PRIVATE_DATA_DESCRIPTION_PATH, index_col='filename').sort_index()
# there is no real target in private data description
dummy_target = {key: 0 for key in description.index}

val_loader = create_dataloader(
    img_dir_path=PRIVATE_DATA_FOLDER_PATH,
    target_map=dummy_target,
    description=description,
    batch_size=BATCH_SIZE,
    preprocessor=preprocess,
    num_load_workers=0,
)


def mark_data_from_one_model(model, device, train_loader):
    model.eval()
    y_real = []
    y_pred = []
    pass_ids = []
    plan_sides = []
    fact_sides = []
    ground_truth = []
    with torch.no_grad():
        for batch in tqdm(train_loader, total=len(train_loader)):
            images = batch['photo'].to(device)
            outputs = model(images).squeeze()
            y_pred.extend(list(outputs.detach().cpu().numpy()))
            pass_ids.extend(batch['pass_id'])
            plan_sides.extend(batch['plan_side'])
            
    return pd.DataFrame.from_dict({
        'pass_id': pass_ids,
        'prediction': y_pred,
        'plan_side': plan_sides,
    })

new_val_prediction = mark_data_from_one_model(model, device, val_loader)

new_train_data = []
dimens = 9
for pass_id, group in new_val_prediction.groupby("pass_id"):
    group = group.sort_values(by="plan_side") 
    prediction_list = np.asarray(list(group['prediction'])+[np.zeros(dimens) for i in range(4-len(group))])[[1, 0, 2, 3]]
    new_train_data.append([pass_id, np.asarray(prediction_list).reshape(dimens*4)])
new_train_data_df = pd.DataFrame(new_train_data, columns=['pass_id', 'prediction'])

X_test = np.zeros((len(new_train_data_df), dimens*8+(dimens*4)**2))
for idx, i in zip(range(len(new_train_data_df)), new_train_data_df.prediction):
    X_test[idx, :] =  np.hstack((i,(i[None,:]*i[:, None]).flatten(), (i**3).flatten()))
    

with open('clf_98.pickle', 'rb') as f:
    clf = pickle.load(f)

y_pred = clf.predict_proba(X_test)[:, 1]
final_solution = pd.DataFrame()
final_solution['pass_id'] = new_train_data_df['pass_id']
final_solution['prediction'] = y_pred

final_solution.to_csv('./predictions.csv')
'''

In [97]:
# build the .zip to submit
import zipfile
import datetime

def make_zip_submission(model_path, solution_script):

    with open('run.py', 'w') as f_run:
        f_run.write(solution_script)

    with open('run.sh', 'w') as f_run_sh:
        f_run_sh.write('python run.py')

    with open('prepare.py', 'w') as f_run:
        f_run.write('print("do nothing")')

    with open('prepare.sh', 'w') as f_run_sh:
        f_run_sh.write('python prepare.py')

    with open('Makefile', 'w') as f_makefile:
        f_makefile.write(
'''prepare:
\tbash prepare.sh
run:
\tbash run.sh
''')

    submission_zip = zipfile.ZipFile(
        f"submission-{datetime.datetime.now()}.zip".replace(':', '-').replace(' ', '-'),
        "w"
    )
    submission_zip.write('./Makefile', compress_type=zipfile.ZIP_DEFLATED)
    submission_zip.write('run.py', compress_type=zipfile.ZIP_DEFLATED)
    submission_zip.write('run.sh', compress_type=zipfile.ZIP_DEFLATED)
    #submission_zip.write('train.py', compress_type=zipfile.ZIP_DEFLATED)
    #submission_zip.write('train.sh', compress_type=zipfile.ZIP_DEFLATED)
    submission_zip.write('prepare.py', compress_type=zipfile.ZIP_DEFLATED)
    submission_zip.write('prepare.sh', compress_type=zipfile.ZIP_DEFLATED)
    submission_zip.write(model_path, compress_type=zipfile.ZIP_DEFLATED)
    submission_zip.write('utils_fixed.py', compress_type=zipfile.ZIP_DEFLATED)
    submission_zip.write('clf_98.pickle', compress_type=zipfile.ZIP_DEFLATED)

    submission_zip.close()

In [98]:
make_zip_submission(model_path='roma_7d_v5.pt', solution_script=solution_script)

In [ ]:
!python run.py